# Reproduce: *When Are Neural Interaction Discoveries Real?*

**To verify every number in the paper: `Runtime > Run all` (top menu). That is the only step.**

No GPU, no datasets, no setup. The first cell downloads the code+results bundle automatically; the second recomputes every reported number from the committed result files and prints a PASS/FAIL line for each. A correct run ends with `42/42 checks passed`. Runs in well under a minute.

(Optional, for re-training from scratch on a GPU, see the per-experiment notebooks in the bundle's `notebooks/` folder. That is not needed to verify the paper's numbers.)

In [ ]:
# === Cell 1: fetch the bundle (auto; nothing to upload) ===
import os, zipfile, urllib.request, sys

REPO_ID = 'ICDM-GNAVAR-EDAE'
WORK = '/content/repro' if os.path.isdir('/content') else os.path.expanduser('~/repro')
os.makedirs(WORK, exist_ok=True)
zip_path = os.path.join(WORK, 'bundle.zip')

# Anonymous GitHub serves an anonymized ZIP of the repo. The download endpoint
# has varied over time, so try the known forms in order.
candidate_urls = [
    f'https://anonymous.4open.science/api/repo/{REPO_ID}/zip',
    f'https://anonymous.4open.science/api/repo/{REPO_ID}/download',
    f'https://anonymous.4open.science/r/{REPO_ID}/zip',
]
ok = False
for url in candidate_urls:
    try:
        print('Trying', url)
        req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
        with urllib.request.urlopen(req, timeout=120) as r, open(zip_path, 'wb') as f:
            f.write(r.read())
        with zipfile.ZipFile(zip_path) as z:
            z.extractall(WORK)
        print('Downloaded and unzipped to', WORK); ok = True; break
    except Exception as e:
        print('  failed:', e)
if not ok:
    print('\nAll auto-download attempts failed. ONE manual step instead:\n'
          'on the Anonymous GitHub page click the Download button (top right),\n'
          'then in Colab use the Files pane (left) to upload the zip into this session,\n'
          'and set REPO_ZIP below to its name and re-run this cell.')
    REPO_ZIP = ''  # e.g. 'ICDM-GNAVAR-EDAE.zip' if you uploaded it manually
    if REPO_ZIP:
        with zipfile.ZipFile(os.path.join('/content', REPO_ZIP)) as z:
            z.extractall(WORK)
        print('Unzipped uploaded file to', WORK)

# locate the repo root (the folder that contains results/ and src/)
ROOT = None
for dirpath, dirnames, filenames in os.walk(WORK):
    if 'results' in dirnames and 'src' in dirnames:
        ROOT = dirpath; break
assert ROOT, 'Could not find the bundle root (a folder containing results/ and src/).'
print('Bundle root:', ROOT)
!pip -q install numpy pandas >/dev/null 2>&1
print('Ready.')

In [ ]:
# === Cell 2: recompute and check every paper number ===
import sys
sys.path.insert(0, ROOT + '/src')
from verifier_core import CHECKS

def _cmp(kind, expected, got, tol):
    if kind == 'exact':  return got == expected
    if kind == 'tol':    return abs(float(got) - float(expected)) <= tol
    if kind == 'range':
        lo, hi = expected
        return (lo <= got[0] and got[1] <= hi) if isinstance(got,(tuple,list)) else (lo <= got <= hi)
    raise ValueError(kind)

def _fmt(v):
    if isinstance(v, float): return f'{v:.4f}'
    if isinstance(v, (tuple, list)): return '(' + ', '.join(_fmt(x) for x in v) + ')'
    return str(v)

n_pass = 0; fails = []; sec = None
for entry in CHECKS:
    section, quantity, kind, expected, fn = entry[:5]
    tol = entry[5] if len(entry) > 5 else 0.0
    if section != sec: print(f'\n[Section {section}]'); sec = section
    try:
        got = fn(ROOT); ok = _cmp(kind, expected, got, tol)
    except Exception as e:
        got = f'ERROR: {e}'; ok = False
    print(f"  [{'PASS' if ok else 'FAIL'}] {quantity:<52} paper={_fmt(expected)}  artifact={_fmt(got)}")
    if ok: n_pass += 1
    else: fails.append((section, quantity, expected, got))

print('\n' + '='*70)
print(f'RESULT: {n_pass}/{len(CHECKS)} checks passed.')
if fails:
    print('FAILED:'); [print(f'  [{s}] {q}: paper={_fmt(e)} artifact={_fmt(g)}') for s,q,e,g in fails]
else:
    print('All paper numbers reproduce from the committed artifacts.')